# Advanced Genetic Mapping: Three-Point Crosses & Interference

## Interactive Learning Module

**Authors:** Susama Kar & Dr. Alok Patel  
**Institution:** Department of Zoology, Kuchinda College, Sambalpur University

---

## What You'll Learn

1. 🧬 How to analyze three-point test crosses
2. 🎯 Gene ordering and map construction
3. 📊 Interference and coefficient of coincidence (COC)
4. 📏 Haldane vs Kosambi mapping functions
5. 🎚️ Interactive exploration of all concepts

---

## Prerequisites

Before starting this notebook, you should understand:
- Poisson distribution of crossovers ✓
- 50% recombination frequency limit ✓
- Basic two-point crosses ✓

*If you haven't completed Notebook 1, please do that first!*


In [ ]:
# Setup and Imports

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import poisson
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox, RadioButtons
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

print("✓ All libraries imported successfully!")
print("\n🎯 Ready to explore advanced genetic mapping!")

---

## 🧬 Part 1: Three-Point Test Cross

### Why Three Points?

**Problem with two-point crosses:**
- Can't distinguish: Far apart on same chromosome vs. Different chromosomes
- Both give 50% RF!

**Solution: Three-point cross**
- Test three genes simultaneously: A, B, C
- Cross: AaBbCc × aabbcc (heterozygote × homozygous recessive)
- Observe 8 possible gamete classes

### The Eight Gamete Classes

1. **Parental types** (most frequent)
   - ABC and abc

2. **Single crossover Region I** (between A and B)
   - AbC and aBc

3. **Single crossover Region II** (between B and C)
   - ABc and abC

4. **Double crossover** (both regions) - RAREST
   - Abc and aBC

---

### 🎚️ Interactive: Gene Order Determination


In [ ]:
@interact(
    rf_ab=FloatSlider(min=0.05, max=0.40, step=0.01, value=0.18, 
                     description='RF(A-B):', style={'description_width': 'initial'}),
    rf_ac=FloatSlider(min=0.05, max=0.40, step=0.01, value=0.30, 
                     description='RF(A-C):', style={'description_width': 'initial'}),
    rf_bc=FloatSlider(min=0.05, max=0.40, step=0.01, value=0.12, 
                     description='RF(B-C):', style={'description_width': 'initial'}),
    show_steps=Checkbox(value=True, description='Show calculation steps')
)
def determine_gene_order(rf_ab, rf_ac, rf_bc, show_steps):
    """
    Interactive gene order determination from RF data
    """
    # Determine which RF is largest
    rf_dict = {'A-B': rf_ab, 'A-C': rf_ac, 'B-C': rf_bc}
    max_pair = max(rf_dict, key=rf_dict.get)
    max_rf = rf_dict[max_pair]
    
    # Determine gene order
    if max_pair == 'A-B':
        order = "C - A - B" if rf_ac < rf_bc else "B - A - C"
        middle = 'A'
        end1, end2 = 'C', 'B' if rf_ac < rf_bc else 'B', 'C'
    elif max_pair == 'A-C':
        order = "B - A - C" if rf_ab < rf_bc else "C - A - B"
        middle = 'A'
        end1, end2 = 'B', 'C' if rf_ab < rf_bc else 'C', 'B'
    else:  # B-C is max
        order = "A - B - C" if rf_ab < rf_ac else "C - B - A"
        middle = 'B'
        end1, end2 = 'A', 'C' if rf_ab < rf_ac else 'C', 'A'
    
    # Check additivity
    if middle == 'A':
        sum_rf = rf_ab + rf_ac if end1 == 'B' else rf_ab + rf_ac
    elif middle == 'B':
        sum_rf = rf_ab + rf_bc
    
    # Convert to map distances
    distances_cm = {k: v*100 for k, v in rf_dict.items()}
    
    # Visualize
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # RF comparison bar plot
    ax = axes[0]
    pairs = list(rf_dict.keys())
    values = [rf_dict[p]*100 for p in pairs]
    colors = ['red' if p == max_pair else 'steelblue' for p in pairs]
    
    bars = ax.bar(pairs, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    ax.set_ylabel('Recombination Frequency (%)', fontsize=12, fontweight='bold')
    ax.set_title('Step 1: Find Maximum RF (these genes are at the ends)', 
                fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Highlight max
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
    
    # Gene map
    ax2 = axes[1]
    ax2.set_xlim(-5, max_rf*120)
    ax2.set_ylim(0, 2)
    
    # Draw chromosome
    total_dist = max_rf * 100
    ax2.plot([0, total_dist], [1, 1], 'k-', linewidth=8, solid_capstyle='round')
    
    # Mark genes based on order
    gene_positions = {}
    gene_colors = {'A': 'red', 'B': 'blue', 'C': 'green'}
    
    if order.startswith('A'):
        gene_positions = {'A': 0, 'B': rf_ab*100, 'C': rf_ac*100}
    elif order.startswith('B'):
        gene_positions = {'B': 0, 'A': rf_ab*100, 'C': rf_bc*100}
    else:  # starts with C
        gene_positions = {'C': 0, 'A': rf_ac*100, 'B': rf_bc*100}
    
    for gene, pos in gene_positions.items():
        ax2.plot([pos, pos], [0.7, 1.3], color=gene_colors[gene], linewidth=5)
        ax2.text(pos, 0.4, gene, fontsize=28, fontweight='bold', 
               ha='center', color=gene_colors[gene])
    
    ax2.set_title(f'Step 2: Gene Order = {order}', fontsize=14, fontweight='bold')
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if show_steps:
        print("\n📊 Gene Order Algorithm:")
        print("═" * 70)
        print(f"\nStep 1: Compare all three RFs")
        for pair, val in rf_dict.items():
            marker = " ← MAXIMUM" if pair == max_pair else ""
            print(f"   RF({pair}) = {val*100:.2f}%{marker}")
        
        print(f"\nStep 2: Identify outer genes")
        outer_genes = max_pair.split('-')
        print(f"   Maximum RF = {max_pair} = {max_rf*100:.2f}%")
        print(f"   → Genes {outer_genes[0]} and {outer_genes[1]} are at the ENDS")
        
        print(f"\nStep 3: Middle gene")
        all_genes = {'A', 'B', 'C'}
        middle_gene = (all_genes - set(outer_genes)).pop()
        print(f"   → Gene {middle_gene} is in the MIDDLE")
        
        print(f"\nStep 4: Final gene order")
        print(f"   → {order}")
        print(f"\n" + "═" * 70)
    
    print(f"\n🎯 Result: Gene order is {order}")
    print(f"   Middle gene: {middle}")
    print(f"   Total map length: {max_rf*100:.2f} cM")

---

## 📊 Part 2: Coefficient of Coincidence (COC)

### What is Interference?

**Biological Reality:** One crossover makes another nearby crossover less likely.

**Why?**
- Crossover machinery changes chromosome structure
- Creates "zone of interference"
- Reduces probability of second crossover nearby

### Measuring Interference

**Coefficient of Coincidence (COC):**

$$COC = \frac{\text{Observed DCO}}{\text{Expected DCO}}$$

**Where:**
- Expected DCO = RF₁ × RF₂ × Total offspring

**Interference (I):**

$$I = 1 - COC$$

**Interpretation:**
- COC = 1.0 → No interference
- COC = 0.5 → 50% reduction in DCOs
- I = 0.5 → 50% interference

---

### 🎚️ Interactive: COC Calculator


In [ ]:
@interact(
    rf1=FloatSlider(min=0.05, max=0.35, step=0.01, value=0.18, 
                   description='RF Region I:', style={'description_width': 'initial'}),
    rf2=FloatSlider(min=0.05, max=0.35, step=0.01, value=0.12, 
                   description='RF Region II:', style={'description_width': 'initial'}),
    total_offspring=IntSlider(min=100, max=2000, step=100, value=1000, 
                             description='Total Offspring:', style={'description_width': 'initial'}),
    observed_dco=IntSlider(min=0, max=50, step=1, value=12, 
                          description='Observed DCO:', style={'description_width': 'initial'})
)
def calculate_coc(rf1, rf2, total_offspring, observed_dco):
    """
    Interactive COC and interference calculator
    """
    # Calculate expected DCO
    expected_dco = rf1 * rf2 * total_offspring
    
    # Calculate COC
    coc = observed_dco / expected_dco if expected_dco > 0 else 0
    
    # Calculate interference
    interference = 1 - coc
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # DCO comparison
    ax = axes[0]
    categories = ['Expected\nDCO', 'Observed\nDCO']
    values = [expected_dco, observed_dco]
    colors = ['lightgreen', 'coral']
    
    bars = ax.bar(categories, values, color=colors, alpha=0.7, 
                  edgecolor='black', linewidth=2)
    ax.set_ylabel('Count', fontsize=12, fontweight='bold')
    ax.set_title('Double Crossover Comparison', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + max(values)*0.02,
                f'{val:.1f}', ha='center', fontweight='bold', fontsize=12)
    
    # COC gauge
    ax2 = axes[1]
    theta = np.linspace(np.pi, 2*np.pi, 100)
    r = 1
    
    # Background arc
    ax2.plot(r * np.cos(theta), r * np.sin(theta), 'k-', linewidth=3)
    
    # COC indicator
    coc_angle = np.pi + coc * np.pi
    ax2.plot([0, r * np.cos(coc_angle)], [0, r * np.sin(coc_angle)], 
            'r-', linewidth=4, marker='o', markersize=12)
    
    # Labels
    ax2.text(r * np.cos(np.pi), r * np.sin(np.pi) - 0.2, '0.0\n(Complete\nInterference)', 
            ha='center', fontsize=9, fontweight='bold')
    ax2.text(r * np.cos(3*np.pi/2), r * np.sin(3*np.pi/2) - 0.3, '0.5', 
            ha='center', fontsize=9, fontweight='bold')
    ax2.text(r * np.cos(2*np.pi), r * np.sin(2*np.pi) - 0.2, '1.0\n(No\nInterference)', 
            ha='center', fontsize=9, fontweight='bold')
    
    ax2.text(0, -0.6, f'COC = {coc:.3f}', ha='center', 
            fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
    
    ax2.set_xlim(-1.5, 1.5)
    ax2.set_ylim(-1, 0.5)
    ax2.set_aspect('equal')
    ax2.axis('off')
    ax2.set_title('Coefficient of Coincidence', fontsize=13, fontweight='bold')
    
    # Interference bar
    ax3 = axes[2]
    
    # Horizontal bar showing interference
    ax3.barh([0], [interference], color='coral', alpha=0.7, 
            edgecolor='black', linewidth=2, height=0.5)
    ax3.barh([0], [1-interference], left=[interference], color='lightgreen', 
            alpha=0.7, edgecolor='black', linewidth=2, height=0.5)
    
    ax3.set_xlim(0, 1)
    ax3.set_ylim(-0.5, 0.5)
    ax3.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax3.set_xticklabels(['0%', '25%', '50%', '75%', '100%'], fontweight='bold')
    ax3.set_yticks([])
    ax3.set_title('Interference Level', fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='x')
    
    # Add text annotation
    ax3.text(interference/2, 0, f'{interference*100:.1f}%\nReduction', 
            ha='center', va='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n📊 Calculation Summary:")
    print("═" * 70)
    print(f"\nInput Data:")
    print(f"   RF Region I:       {rf1*100:.2f}%")
    print(f"   RF Region II:      {rf2*100:.2f}%")
    print(f"   Total offspring:   {total_offspring}")
    print(f"   Observed DCO:      {observed_dco}")
    
    print(f"\nCalculations:")
    print(f"   Expected DCO = {rf1:.3f} × {rf2:.3f} × {total_offspring} = {expected_dco:.2f}")
    print(f"   COC = {observed_dco} / {expected_dco:.2f} = {coc:.3f}")
    print(f"   Interference = 1 - {coc:.3f} = {interference:.3f} ({interference*100:.1f}%)")
    
    print(f"\nInterpretation:")
    if interference < 0.2:
        print(f"   → Weak interference: Crossovers nearly independent")
    elif interference < 0.5:
        print(f"   → Moderate interference: Typical for most organisms")
    elif interference < 0.8:
        print(f"   → Strong interference: One CO greatly reduces nearby COs")
    else:
        print(f"   → Very strong interference: Second CO very rare")
    
    print("\n" + "═" * 70)

---

## 📏 Part 3: Mapping Functions

### Why Mapping Functions?

**Problem:** RF underestimates true map distance for long intervals
- Double crossovers "cancel out"
- Need to correct for undetected DCOs

**Solution:** Mapping functions

### Two Common Mapping Functions

**1. Haldane (1919) - Assumes NO interference:**

$$d = -\frac{1}{2} \ln(1 - 2r)$$

**2. Kosambi (1944) - Accounts for interference:**

$$d = \frac{1}{4} \ln\left(\frac{1 + 2r}{1 - 2r}\right)$$

### When to Use Which?

- **Haldane:** Organisms with low interference (yeast, fungi)
- **Kosambi:** Most eukaryotes (fish, plants, animals)

---

### 🎚️ Interactive: Compare Mapping Functions


In [ ]:
@interact(
    max_rf=FloatSlider(min=0.1, max=0.49, step=0.01, value=0.35, 
                      description='Max RF:', style={'description_width': 'initial'}),
    show_haldane=Checkbox(value=True, description='Show Haldane'),
    show_kosambi=Checkbox(value=True, description='Show Kosambi'),
    show_linear=Checkbox(value=True, description='Show linear (RF × 100)')
)
def compare_mapping_functions(max_rf, show_haldane, show_kosambi, show_linear):
    """
    Interactive comparison of Haldane and Kosambi mapping functions
    """
    r = np.linspace(0.01, max_rf, 200)
    
    # Haldane function
    d_haldane = -0.5 * np.log(1 - 2*r) * 100  # in cM
    
    # Kosambi function
    d_kosambi = 0.25 * np.log((1 + 2*r) / (1 - 2*r)) * 100  # in cM
    
    # Linear (no correction)
    d_linear = r * 100
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Main comparison
    ax = axes[0]
    
    if show_linear:
        ax.plot(r*100, d_linear, 'g--', linewidth=2, label='Linear (no correction)', alpha=0.7)
    if show_haldane:
        ax.plot(r*100, d_haldane, 'b-', linewidth=3, label='Haldane (no interference)')
    if show_kosambi:
        ax.plot(r*100, d_kosambi, 'r-', linewidth=3, label='Kosambi (with interference)')
    
    ax.set_xlabel('Recombination Frequency (%)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Map Distance (cM)', fontsize=13, fontweight='bold')
    ax.set_title('Mapping Functions Comparison', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # Difference plot
    ax2 = axes[1]
    
    if show_haldane and show_kosambi:
        diff = d_haldane - d_kosambi
        ax2.plot(r*100, diff, 'purple', linewidth=3)
        ax2.fill_between(r*100, 0, diff, alpha=0.3, color='purple')
        ax2.set_xlabel('Recombination Frequency (%)', fontsize=13, fontweight='bold')
        ax2.set_ylabel('Difference (Haldane - Kosambi) cM', fontsize=12, fontweight='bold')
        ax2.set_title('Impact of Interference Correction', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.axhline(y=0, color='k', linestyle='-', linewidth=1)
        
        # Annotate max difference
        max_diff_idx = np.argmax(diff)
        max_diff_rf = r[max_diff_idx] * 100
        max_diff_val = diff[max_diff_idx]
        ax2.plot(max_diff_rf, max_diff_val, 'ro', markersize=10)
        ax2.annotate(f'Max difference\nRF={max_diff_rf:.1f}%\n{max_diff_val:.1f} cM',
                    xy=(max_diff_rf, max_diff_val), xytext=(20, 20),
                    textcoords='offset points',
                    bbox=dict(boxstyle='round', fc='yellow', alpha=0.8),
                    arrowprops=dict(arrowstyle='->', lw=2),
                    fontsize=10, fontweight='bold')
    else:
        ax2.text(0.5, 0.5, 'Enable both Haldane and Kosambi\nto see difference', 
                ha='center', va='center', transform=ax2.transAxes,
                fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat'))
        ax2.set_xticks([])
        ax2.set_yticks([])
    
    plt.tight_layout()
    plt.show()
    
    # Print comparison table
    test_rfs = [0.10, 0.20, 0.30, max_rf]
    
    print("\n📊 Map Distance Comparison Table:")
    print("═" * 80)
    print(f"{'RF (%)':<10} {'Linear (cM)':<15} {'Haldane (cM)':<15} {'Kosambi (cM)':<15} {'Difference':<15}")
    print("═" * 80)
    
    for rf in test_rfs:
        if rf <= max_rf:
            lin = rf * 100
            hal = -0.5 * np.log(1 - 2*rf) * 100
            kos = 0.25 * np.log((1 + 2*rf) / (1 - 2*rf)) * 100
            diff = hal - kos
            print(f"{rf*100:<10.1f} {lin:<15.2f} {hal:<15.2f} {kos:<15.2f} {diff:<15.2f}")
    
    print("═" * 80)
    print("\n💡 Key Insight: As RF increases, the difference between functions grows!")
    print("   → Haldane overestimates distance (no interference assumption)")
    print("   → Kosambi is more accurate for most organisms")

---

## 🎯 Summary & Key Takeaways

### What You've Learned:

1. ✅ **Three-point crosses** resolve linkage ambiguity
   - Eight gamete classes
   - Gene ordering from RF data
   - Maximum RF identifies outer genes

2. ✅ **Interference** is biological reality
   - COC measures deviation from independence
   - Interference = 1 - COC
   - Typical organisms show I = 0.3-0.7

3. ✅ **Mapping functions** correct for multiple crossovers
   - Haldane: No interference (fungi, yeast)
   - Kosambi: With interference (most eukaryotes)
   - Difference increases with distance

4. ✅ **Practical applications**
   - Choose correct mapping function
   - Account for interference
   - Build accurate genetic maps

### Complete Learning Path:

✓ Notebook 1: Poisson distribution, 50% limit  
✓ Notebook 2: Three-point crosses, interference  
➡️ Next: Apply to your own research data!

---

## 📚 References

- Haldane, J.B.S. (1919). *Journal of Genetics*
- Kosambi, D.D. (1944). *Annals of Eugenics*
- Sturtevant, A.H. (1913). *Journal of Experimental Zoology*

---

**Authors:** Susama Kar & Dr. Alok Patel  
**Institution:** Kuchinda College, Sambalpur University  
**License:** CC BY 4.0  
**Repository:** github.com/The-Pattern-Hunter/principles-of-genetics-interactive
